# IO Benchmark - 2026-04-07

Compare the four write targets users actually choose between when persisting
an LBM dataset. Each format writes the full dataset end-to-end, one pass each:

| format             | layout                                   | compression        |
| ------------------ | ---------------------------------------- | ------------------ |
| `tiff`             | single ImageJ volumetric hyperstack      | none               |
| `bin`              | suite2p per-plane `plane0N/data_raw.bin` | none               |
| `zarr-none`        | zarr v3 sharded, 1-frame inner chunks    | none               |
| `zarr-zstd-3`      | zarr v3 sharded, 1-frame inner chunks    | zstd level 3       |
| `zarr-blosc-zstd`  | zarr v3 sharded, 1-frame inner chunks    | blosc-zstd level 3 |

A prior run on this box showed writes to `E:` are disk-bound at ~33 MiB/s, so
the three uncompressed formats should cluster together (any delta between them
is format overhead — headers, per-plane file creation, zarr shard index), and
the compressed zarr variants should finish faster *because they write fewer
bytes to disk*, not because of CPU differences.

Methodology:

- One untimed warmup write (zarr-none) primes the OS page cache for source reads.
- The 5 conditions run in shuffled order (seeded) so residual cache drift doesn't
  align with the format axis.
- Two figures at the end: wall time per format, on-disk size per format.

In [1]:
import gc
import os
import platform
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mbo_utilities
from mbo_utilities import imread

print(f'mbo_utilities: {mbo_utilities.__version__}')
print(f'Python: {platform.python_version()}')
print(f'CPU cores: {os.cpu_count()}')
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f'RAM: {mem.total / 1024**3:.1f} GB total, {mem.available / 1024**3:.1f} GB available')
except ImportError:
    pass

mbo_utilities: 2.7.0
Python: 3.12.9
CPU cores: 32
RAM: 127.8 GB total, 101.4 GB available


In [2]:
import random

# paths
SOURCE_PATH = Path(r'E:\datasets\lbm\2025-07-27_mk355-kbarber\raw')
OUTPUT_ROOT = Path(r'E:\datasets\lbm\2025-07-27_mk355-kbarber\io_benchmark')

# conditions: (label, ext, kwargs for arr._imwrite)
# zarr target_chunk_mb=128 is in the knee region from the prior codec sweep —
# not the variable under test here.
CONDITIONS = [
    ('tiff',            '.tiff', {}),
    ('bin',             '.bin',  {}),
    ('zarr-none',       '.zarr', {'compressor': 'none',       'compression_level': 0, 'target_chunk_mb': 128, 'sharded': True}),
    ('zarr-zstd-3',     '.zarr', {'compressor': 'zstd',       'compression_level': 3, 'target_chunk_mb': 128, 'sharded': True}),
    ('zarr-blosc-zstd', '.zarr', {'compressor': 'blosc-zstd', 'compression_level': 3, 'target_chunk_mb': 128, 'sharded': True}),
]

OVERWRITE = True
SEED = 20260407  # reproducible shuffle order

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# shuffle the order so cache drift doesn't align with the format axis
order = list(range(len(CONDITIONS)))
rng = random.Random(SEED)
rng.shuffle(order)
shuffled = [CONDITIONS[i] for i in order]

print(f'Source: {SOURCE_PATH}')
print(f'Output: {OUTPUT_ROOT}')
print(f'Conditions: {len(CONDITIONS)} formats, one pass each (+1 warmup)')
print('Run order:', [c[0] for c in shuffled])

Source: E:\datasets\lbm\2025-07-27_mk355-kbarber\raw
Output: E:\datasets\lbm\2025-07-27_mk355-kbarber\io_benchmark
Conditions: 5 formats, one pass each (+1 warmup)
Run order: ['bin', 'tiff', 'zarr-zstd-3', 'zarr-none', 'zarr-blosc-zstd']


In [3]:
arr = imread(str(SOURCE_PATH))
print(f'type: {type(arr).__name__}')
print(f'shape (5D TCZYX): {arr.shape5d}')
print(f'dtype: {arr.dtype}')

raw_bytes = int(np.prod(arr.shape5d)) * np.dtype(arr.dtype).itemsize
print(f'raw payload: {raw_bytes / 1024**3:.2f} GiB (uncompressed, full TCZYX)')

Counting frames:   0%|          | 0/2 [00:00<?, ?it/s]

type: LBMArray
shape (5D TCZYX): (1574, 1, 14, 550, 448)
dtype: int16
raw payload: 10.11 GiB (uncompressed, full TCZYX)


In [4]:
def dir_size(path: Path) -> int:
    """Total bytes occupied by everything under `path` (or the file itself)."""
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = Path(root) / f
            try:
                total += fp.stat().st_size
            except OSError:
                pass
    return total


def run_one(arr, label: str, ext: str, kwargs: dict) -> dict:
    """Write the full dataset in the given format and measure time + size."""
    out_dir = OUTPUT_ROOT / label
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)

    gc.collect()
    t0 = time.perf_counter()
    arr._imwrite(
        outpath=out_dir,
        ext=ext,
        overwrite=OVERWRITE,
        show_progress=False,
        **kwargs,
    )
    t1 = time.perf_counter()
    elapsed = t1 - t0

    on_disk = dir_size(out_dir)
    return {
        'format': label,
        'ext': ext,
        'time_s': elapsed,
        'on_disk_bytes': on_disk,
        'on_disk_mib': on_disk / 1024**2,
        'on_disk_gib': on_disk / 1024**3,
        'raw_mibps': (raw_bytes / 1024**2) / elapsed,
        'disk_mibps': (on_disk / 1024**2) / elapsed,
        'compression_ratio': raw_bytes / on_disk if on_disk > 0 else float('nan'),
        'out_dir': str(out_dir),
    }

In [5]:
# warmup: one untimed zarr-none write to prime the OS page cache on the source
print('Warmup write (untimed, zarr-none) ...', end=' ', flush=True)
warmup_dir = OUTPUT_ROOT / 'warmup'
if warmup_dir.exists():
    shutil.rmtree(warmup_dir)
warmup_dir.mkdir()
gc.collect()
_t0 = time.perf_counter()
arr._imwrite(
    outpath=warmup_dir,
    ext='.zarr',
    overwrite=True,
    compressor='none',
    compression_level=0,
    target_chunk_mb=128,
    sharded=True,
    show_progress=False,
)
print(f'{time.perf_counter() - _t0:.1f} s')
shutil.rmtree(warmup_dir)

# timed run — shuffled order
results = []
total = len(shuffled)
for i, (label, ext, kwargs) in enumerate(shuffled, start=1):
    print(f'[{i}/{total}] {label:<18} ...', end=' ', flush=True)
    try:
        row = run_one(arr, label, ext, kwargs)
    except Exception as e:
        print(f'FAILED: {type(e).__name__}: {e}')
        results.append({
            'format': label,
            'ext': ext,
            'time_s': float('nan'),
            'on_disk_bytes': 0,
            'on_disk_mib': float('nan'),
            'on_disk_gib': float('nan'),
            'raw_mibps': float('nan'),
            'disk_mibps': float('nan'),
            'compression_ratio': float('nan'),
            'error': str(e),
        })
        continue
    print(
        f"{row['time_s']:6.1f} s | {row['on_disk_gib']:5.2f} GiB | "
        f"raw {row['raw_mibps']:5.1f} MiB/s | ratio {row['compression_ratio']:.2f}x"
    )
    results.append(row)

df = pd.DataFrame(results)
# display rows in the CONDITIONS order (not shuffled run order)
format_order = [c[0] for c in CONDITIONS]
df['format'] = pd.Categorical(df['format'], categories=format_order, ordered=True)
df = df.sort_values('format').reset_index(drop=True)
df

Warmup write (untimed, zarr-none) ... 324.0 s
[1/5] bin                ...  319.3 s | 10.11 GiB | raw  32.4 MiB/s | ratio 1.00x
[2/5] tiff               ...  308.7 s | 10.12 GiB | raw  33.6 MiB/s | ratio 1.00x
[3/5] zarr-zstd-3        ...  320.6 s |  8.17 GiB | raw  32.3 MiB/s | ratio 1.24x
[4/5] zarr-none          ...  317.9 s | 10.11 GiB | raw  32.6 MiB/s | ratio 1.00x
[5/5] zarr-blosc-zstd    ...  325.9 s |  7.82 GiB | raw  31.8 MiB/s | ratio 1.29x


,format,ext,time_s,on_disk_bytes,on_disk_mib,on_disk_gib,raw_mibps,disk_mibps,compression_ratio,out_dir
0,tiff,.tiff,308.650308,10865457532,10362.107784,10.119246,33.553423,33.572323,0.999437,E:\datasets\lbm\2025-07-27_mk355-kbarber\io_be...
1,bin,.bin,319.324657,10859357458,10356.290300,10.113565,32.431803,32.431853,0.999998,E:\datasets\lbm\2025-07-27_mk355-kbarber\io_be...
2,zarr-none,.zarr,317.857331,10859406858,10356.337412,10.113611,32.581518,32.581716,0.999994,E:\datasets\lbm\2025-07-27_mk355-kbarber\io_be...
3,zarr-zstd-3,.zarr,320.552046,8770923388,8364.604366,8.168559,32.307622,26.094372,1.238107,E:\datasets\lbm\2025-07-27_mk355-kbarber\io_be...
4,zarr-blosc-zstd,.zarr,325.929425,8397448200,8008.430672,7.820733,31.774592,24.571058,1.293172,E:\datasets\lbm\2025-07-27_mk355-kbarber\io_be...


In [ ]:
# persist the results table next to the outputs
csv_path = OUTPUT_ROOT / '2026-04-07_io_results.csv'
df.to_csv(csv_path, index=False)
print(f'Saved results: {csv_path}')

## Plots

Two simple figures:

1. **Wall time per format** — how long the full dataset takes to write
2. **On-disk size per format** — how much space the written output occupies

A red dashed line in figure 2 marks the raw uncompressed payload size so you
can read the compression ratio by eye.

In [ ]:
ok = df.dropna(subset=['time_s']).copy()

raw_mib = raw_bytes / 1024**2
raw_gib = raw_bytes / 1024**3

# color per format — viridis across the x-axis
labels = list(ok['format'])
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(labels)))

# figure 1: wall time
fig1, ax1 = plt.subplots(figsize=(8, 4.5))
bars1 = ax1.bar(labels, ok['time_s'].values, color=colors, edgecolor='k', linewidth=0.6)
ax1.set_ylabel('write time (s)')
ax1.set_title('How fast — wall time per format')
ax1.grid(axis='y', alpha=0.3)
for bar, t_s, rate in zip(bars1, ok['time_s'], ok['raw_mibps']):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{t_s:.0f}s\n{rate:.0f} MiB/s',
        ha='center', va='bottom', fontsize=8,
    )
ax1.set_ylim(0, ok['time_s'].max() * 1.15)
fig1.suptitle(
    f'{SOURCE_PATH.name}  |  shape {tuple(arr.shape5d)}  |  {raw_gib:.2f} GiB raw',
    fontsize=10,
)
fig1.tight_layout()
fig1.savefig(OUTPUT_ROOT / '2026-04-07_io_time.png', dpi=120, bbox_inches='tight')
plt.show()

# figure 2: on-disk size
fig2, ax2 = plt.subplots(figsize=(8, 4.5))
bars2 = ax2.bar(labels, ok['on_disk_gib'].values, color=colors, edgecolor='k', linewidth=0.6)
ax2.axhline(raw_gib, color='crimson', linestyle='--', linewidth=1.2,
            label=f'raw ({raw_gib:.2f} GiB)')
ax2.set_ylabel('on-disk size (GiB)')
ax2.set_title('How much written — on-disk size per format')
ax2.grid(axis='y', alpha=0.3)
ax2.legend(fontsize=8, loc='upper right')
for bar, gib, ratio in zip(bars2, ok['on_disk_gib'], ok['compression_ratio']):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{gib:.2f} GiB\n{ratio:.2f}x',
        ha='center', va='bottom', fontsize=8,
    )
ax2.set_ylim(0, max(raw_gib, ok['on_disk_gib'].max()) * 1.15)
fig2.suptitle(
    f'{SOURCE_PATH.name}  |  shape {tuple(arr.shape5d)}  |  {raw_gib:.2f} GiB raw',
    fontsize=10,
)
fig2.tight_layout()
fig2.savefig(OUTPUT_ROOT / '2026-04-07_io_size.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# summary — actionable recommendation per goal
if not ok.empty:
    fastest = ok.loc[ok['time_s'].idxmin()]
    smallest = ok.loc[ok['on_disk_gib'].idxmin()]

    def _fmt(row):
        return (f"{row['format']:<18} -> {row['time_s']:6.1f} s  |  "
                f"{row['on_disk_gib']:5.2f} GiB  |  ratio {row['compression_ratio']:.2f}x  "
                f"|  {row['raw_mibps']:5.1f} MiB/s raw")

    print('Fastest write:')
    print(f'  {_fmt(fastest)}')
    print()
    print('Smallest on-disk:')
    print(f'  {_fmt(smallest)}')
    print()
    print(f'Raw payload: {raw_gib:.2f} GiB  |  source shape: {tuple(arr.shape5d)}  |  dtype: {arr.dtype}')
    print()
    print('Per-format summary:')
    for _, row in ok.iterrows():
        print(f'  {_fmt(row)}')